In [ ]:
# Edge Arabic TTS — HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel
# GPU: Settings → Accelerator → GPU T4

import os
if not os.path.exists("/kaggle/working/Sarj"):
    !git clone https://github.com/OmarHammemi/Sarj.git /kaggle/working/Sarj

%cd /kaggle/working/Sarj
!pip install -q -r requirements.txt datasets librosa soundfile pyyaml tqdm

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%%writefile data/audio_utils.py
"""Shared librosa mel + Griffin-Lim (train and synth must match)."""
from __future__ import annotations
import numpy as np

def mel_spectrogram(wav: np.ndarray, cfg: dict) -> np.ndarray:
    import librosa
    a = cfg["audio"]
    mel = librosa.feature.melspectrogram(
        y=wav, sr=a["sample_rate"], n_fft=a["n_fft"],
        hop_length=a["hop_length"], win_length=a["win_length"],
        n_mels=a["n_mels"], fmin=a["fmin"], fmax=a["fmax"], power=1.0,
    )
    return np.log(np.maximum(mel, 1e-5)).astype(np.float32)

def mel_to_wav(mel: np.ndarray, cfg: dict, n_iters: int = 64) -> np.ndarray:
    import librosa
    a = cfg["audio"]
    stft = librosa.feature.inverse.mel_to_stft(
        np.exp(mel), sr=a["sample_rate"], n_fft=a["n_fft"],
        fmin=a["fmin"], fmax=a["fmax"], power=1.0,
    )
    return librosa.griffinlim(
        stft, n_iter=n_iters, hop_length=a["hop_length"],
        win_length=a["win_length"], n_fft=a["n_fft"],
    )




In [ ]:
%%writefile data/download_hesham.py
"""Download + filter HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel (~1 hour)."""
from __future__ import annotations
import argparse, json, re, sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from datasets import load_dataset
from tqdm import tqdm

RAW_DIR = ROOT / "data" / "raw" / "wav"
DATASET_ID = "HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel"

def is_arabic(text: str, min_ratio: float = 0.6) -> bool:
    text = text.strip()
    if not text:
        return False
    ar = len(re.findall(r"[\u0600-\u06FF]", text))
    return ar / max(len(text.replace(" ", "")), 1) >= min_ratio

def download_filtered(
    output_csv: Path,
    max_hours: float = 1.0,
    min_dur: float = 2.0,
    max_dur: float = 12.0,
    min_chars: int = 10,
    max_chars: int = 180,
    target_sr: int = 22050,
):
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    max_seconds = max_hours * 3600.0

    print(f"Streaming {DATASET_ID} ...")
    ds = load_dataset(DATASET_ID, split="train", streaming=True)

    rows, total_sec = [], 0.0
    skipped = {"duration": 0, "text": 0, "arabic": 0, "empty": 0, "no_audio": 0}

    for item in tqdm(ds, desc="Filter+download"):
        # use diacritized text (better for TTS)
        text = str(item.get("text") or item.get("text_stripped") or "").strip()
        dur_meta = float(item.get("audio_duration_s") or 0)

        if dur_meta and (dur_meta < min_dur or dur_meta > max_dur):
            skipped["duration"] += 1
            continue
        if len(text) < min_chars or len(text) > max_chars:
            skipped["text"] += 1
            continue
        if not is_arabic(text):
            skipped["arabic"] += 1
            continue

        audio = item.get("audio")
        if audio is None:
            skipped["no_audio"] += 1
            continue

        wav = np.asarray(audio["array"], dtype=np.float32)
        sr_in = int(audio["sampling_rate"])
        if wav.size == 0:
            skipped["empty"] += 1
            continue

        if sr_in != target_sr:
            wav = librosa.resample(wav, orig_sr=sr_in, target_sr=target_sr)

        dur = len(wav) / target_sr
        if dur < min_dur or dur > max_dur:
            skipped["duration"] += 1
            continue

        if total_sec + dur > max_seconds and rows:
            break

        clip_id = f"hesham_{len(rows):05d}"
        out = RAW_DIR / f"{clip_id}.wav"
        sf.write(out, wav, target_sr)

        rows.append({
            "id": clip_id,
            "wav_path": str(out.relative_to(ROOT)),
            "text": text,
            "duration_sec": round(dur, 3),
            "source_dataset": DATASET_ID,
            "gender": item.get("gender", ""),
            "voice": item.get("voice", ""),
        })
        total_sec += dur
        if total_sec >= max_seconds:
            break

    if not rows:
        raise RuntimeError(f"No clips passed filters. Skipped: {skipped}")

    meta = pd.DataFrame(rows)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    meta.to_csv(output_csv, index=False)

    summary = {
        "clips": len(meta),
        "hours": round(meta["duration_sec"].sum() / 3600, 3),
        "avg_dur": round(meta["duration_sec"].mean(), 2),
        "skipped": skipped,
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f"Saved -> {output_csv}")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--output", default=str(ROOT / "data" / "metadata.csv"))
    p.add_argument("--max-hours", type=float, default=1.0)
    p.add_argument("--min-dur", type=float, default=2.0)
    p.add_argument("--max-dur", type=float, default=12.0)
    a = p.parse_args()
    download_filtered(Path(a.output), max_hours=a.max_hours,
                      min_dur=a.min_dur, max_dur=a.max_dur)

In [ ]:
%%writefile data/preprocess.py
from __future__ import annotations
import argparse, json, sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import librosa, numpy as np, pandas as pd, soundfile as sf, yaml
from tqdm import tqdm
from data.audio_utils import mel_spectrogram
from data.text_normalize import build_vocab, normalize_arabic, save_vocab

def char_durations(n, mel_frames):
    if n <= 0:
        return np.array([], dtype=np.int64)
    base, rem = mel_frames // n, mel_frames - (mel_frames // n) * n
    d = np.full(n, base, dtype=np.int64)
    d[:rem] += 1
    d = np.maximum(d, 1)
    diff = int(d.sum()) - mel_frames
    i = 0
    while diff > 0 and i < n:
        if d[n - 1 - i] > 1:
            d[n - 1 - i] -= 1
            diff -= 1
        i += 1
    while diff < 0:
        d[diff % n] += 1
        diff += 1
    return d

def preprocess(config_path, metadata_path):
    cfg = yaml.safe_load(open(config_path))
    a, sr = cfg["audio"], cfg["audio"]["sample_rate"]
    pw = ROOT / "data" / "processed" / "wav"
    pm = ROOT / "data" / "processed" / "mel"
    dd = ROOT / "data" / "durations"
    for d in (pw, pm, dd):
        d.mkdir(parents=True, exist_ok=True)

    rows = []
    for _, row in tqdm(pd.read_csv(metadata_path).iterrows(), desc="Preprocess"):
        wav, _ = librosa.load(ROOT / row["wav_path"], sr=sr, mono=True)
        wav, _ = librosa.effects.trim(wav, top_db=a["trim_top_db"])
        if len(wav) == 0:
            continue
        peak = np.max(np.abs(wav))
        if peak > 0:
            wav = wav / peak * 0.95
        dur = len(wav) / sr
        if dur < a["min_duration"] or dur > a["max_duration"]:
            continue
        text = normalize_arabic(str(row["text"]))
        if not text:
            continue
        cid = row["id"]
        sf.write(pw / f"{cid}.wav", wav, sr)
        mel = mel_spectrogram(wav, cfg)
        np.save(pm / f"{cid}.npy", mel)
        np.save(dd / f"{cid}.npy", char_durations(len(text), mel.shape[1]))
        rows.append({
            "id": cid,
            "wav_path": str((pw / f"{cid}.wav").relative_to(ROOT)),
            "mel_path": str((pm / f"{cid}.npy").relative_to(ROOT)),
            "duration_path": str((dd / f"{cid}.npy").relative_to(ROOT)),
            "text": text,
            "duration_sec": round(dur, 3),
            "mel_frames": mel.shape[1],
            "num_chars": len(text),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No clips survived preprocessing.")

    df = df.sample(frac=1, random_state=cfg["dataset"]["seed"]).reset_index(drop=True)
    s = int(len(df) * cfg["dataset"]["train_split"])
    df.iloc[:s].to_csv(ROOT / "data" / "train.csv", index=False)
    df.iloc[s:].to_csv(ROOT / "data" / "val.csv", index=False)
    save_vocab(build_vocab(df.iloc[:s]["text"].tolist()), ROOT / "data" / "vocab.json")
    print(json.dumps({
        "clips": len(df), "train": s, "val": len(df) - s,
        "hours": round(df["duration_sec"].sum() / 3600, 3),
        "vocab_size": len(json.load(open(ROOT / "data" / "vocab.json"))),
    }, indent=2))

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--config", default=str(ROOT / "configs" / "default.yaml"))
    p.add_argument("--metadata", default=str(ROOT / "data" / "metadata.csv"))
    a = p.parse_args()
    preprocess(Path(a.config), Path(a.metadata))

In [ ]:
%%writefile synthesize.py
from __future__ import annotations
import argparse
from pathlib import Path
import numpy as np, soundfile as sf, torch, yaml
from data.audio_utils import mel_to_wav
from data.text_normalize import load_vocab, text_to_ids
from model.fastspeech import FastSpeech2

ROOT = Path(__file__).resolve().parent

def load_model(path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    m = FastSpeech2(
        vocab_size=ckpt["vocab_size"],
        n_mels=cfg["audio"]["n_mels"],
        d_model=cfg["model"]["d_model"],
        n_heads=cfg["model"]["n_heads"],
        encoder_layers=cfg["model"]["encoder_layers"],
        decoder_layers=cfg["model"]["decoder_layers"],
        ffn_dim=cfg["model"]["ffn_dim"],
        kernel_size=cfg["model"]["kernel_size"],
        dropout=cfg["model"]["dropout"],
        use_postnet=cfg["model"].get("use_postnet", True),
        max_seq_len=cfg["model"].get("max_seq_len", 512),
    )
    m.load_state_dict(ckpt["model"])
    m.to(device)
    m.eval()
    return m, cfg

def synthesize(text, checkpoint, config_path, output, device_name="cpu"):
    cfg = yaml.safe_load(open(config_path))
    device = torch.device(device_name if torch.cuda.is_available() or device_name == "cpu" else "cpu")
    model, ckpt_cfg = load_model(checkpoint, device)
    ids = text_to_ids(text, load_vocab(ROOT / "data" / "vocab.json"), add_eos=True)
    tokens = torch.tensor([ids], dtype=torch.long, device=device)
    lengths = torch.tensor([len(ids)], dtype=torch.long, device=device)
    with torch.no_grad():
        mel = model.infer(tokens, lengths).cpu().numpy()
    wav = mel_to_wav(mel, ckpt_cfg, cfg["synthesis"].get("griffin_lim_iters", 64))
    peak = np.max(np.abs(wav)) if wav.size else 1.0
    if peak > 0:
        wav = wav / peak * 0.95
    output.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output, wav, ckpt_cfg["audio"]["sample_rate"])
    print(f"Saved {output} ({len(wav) / ckpt_cfg['audio']['sample_rate']:.2f}s)")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--text", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--checkpoint", default=str(ROOT / "checkpoints" / "best.pt"))
    p.add_argument("--config", default=str(ROOT / "configs" / "default.yaml"))
    p.add_argument("--device", default="cpu", choices=["cuda", "cpu"])
    a = p.parse_args()
    synthesize(a.text, Path(a.checkpoint), Path(a.config), Path(a.out), a.device)

In [ ]:
%%writefile model/variance_adaptor.py
from __future__ import annotations
import torch, torch.nn as nn

class ConvNormBlock(nn.Module):
    def __init__(self, d_model, kernel_size=3, dropout=0.1):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size // 2)
        self.norm = nn.LayerNorm(d_model)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.conv(x).transpose(1, 2)
        x = self.drop(self.act(self.norm(x)))
        return x.transpose(1, 2)

class DurationPredictor(nn.Module):
    def __init__(self, d_model, kernel_size=3, dropout=0.1):
        super().__init__()
        self.block1 = ConvNormBlock(d_model, kernel_size, dropout)
        self.block2 = ConvNormBlock(d_model, kernel_size, dropout)
        self.proj = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.block2(self.block1(x.transpose(1, 2))).transpose(1, 2)
        return self.proj(h).squeeze(-1)

def length_regulator(encoded, durations):
    outs, lens = [], []
    for b in range(encoded.size(0)):
        reps = []
        for t in range(encoded.size(1)):
            d = int(torch.round(durations[b, t]).clamp(min=0).item())
            if d > 0:
                reps.append(encoded[b, t].unsqueeze(0).repeat(d, 1))
        o = torch.cat(reps, dim=0) if reps else encoded.new_zeros((1, encoded.size(-1)))
        outs.append(o)
        lens.append(o.size(0))
    ml = max(lens)
    pad = encoded.new_zeros(encoded.size(0), ml, encoded.size(-1))
    for b, o in enumerate(outs):
        pad[b, : o.size(0)] = o
    return pad, torch.tensor(lens, device=encoded.device, dtype=torch.long)

In [ ]:
%%writefile data/download_hesham.py
"""Download + filter HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel (~1 hour)."""
from __future__ import annotations
import argparse, io, json, re, sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from datasets import load_dataset
from tqdm import tqdm

RAW_DIR = ROOT / "data" / "raw" / "wav"
DATASET_ID = "HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel"

def is_arabic(text: str, min_ratio: float = 0.6) -> bool:
    text = text.strip()
    if not text:
        return False
    ar = len(re.findall(r"[\u0600-\u06FF]", text))
    return ar / max(len(text.replace(" ", "")), 1) >= min_ratio

def decode_audio(audio_obj) -> tuple[np.ndarray, int] | None:
    """Handle HF formats: {array,sr}, {bytes,path}, or raw bytes."""
    if audio_obj is None:
        return None

    # Standard decoded Audio feature
    if isinstance(audio_obj, dict):
        if "array" in audio_obj and audio_obj["array"] is not None:
            wav = np.asarray(audio_obj["array"], dtype=np.float32)
            sr = int(audio_obj.get("sampling_rate") or 22050)
            return wav, sr

        # Parquet binary format (this dataset)
        if audio_obj.get("bytes"):
            try:
                wav, sr = sf.read(io.BytesIO(audio_obj["bytes"]))
                return np.asarray(wav, dtype=np.float32), int(sr)
            except Exception:
                pass

        if audio_obj.get("path"):
            try:
                wav, sr = sf.read(audio_obj["path"])
                return np.asarray(wav, dtype=np.float32), int(sr)
            except Exception:
                pass

    if isinstance(audio_obj, (bytes, bytearray)):
        wav, sr = sf.read(io.BytesIO(audio_obj))
        return np.asarray(wav, dtype=np.float32), int(sr)

    return None

def download_filtered(
    output_csv: Path,
    max_hours: float = 1.0,
    min_dur: float = 2.0,
    max_dur: float = 12.0,
    min_chars: int = 10,
    max_chars: int = 180,
    target_sr: int = 22050,
):
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    max_seconds = max_hours * 3600.0

    print(f"Streaming {DATASET_ID} ...")
    ds = load_dataset(DATASET_ID, split="train", streaming=True)

    rows, total_sec = [], 0.0
    skipped = {
        "duration": 0, "text": 0, "arabic": 0,
        "empty": 0, "no_audio": 0, "decode_fail": 0,
    }

    pbar = tqdm(desc="Filter+download")
    for item in ds:
        pbar.update(1)

        text = str(item.get("text") or item.get("text_stripped") or "").strip()
        dur_meta = float(item.get("audio_duration_s") or 0)

        if dur_meta and (dur_meta < min_dur or dur_meta > max_dur):
            skipped["duration"] += 1
            continue
        if len(text) < min_chars or len(text) > max_chars:
            skipped["text"] += 1
            continue
        if not is_arabic(text):
            skipped["arabic"] += 1
            continue

        decoded = decode_audio(item.get("audio"))
        if decoded is None:
            skipped["decode_fail"] += 1
            continue

        wav, sr_in = decoded
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if wav.size == 0:
            skipped["empty"] += 1
            continue

        if sr_in != target_sr:
            wav = librosa.resample(wav, orig_sr=sr_in, target_sr=target_sr)

        dur = len(wav) / target_sr
        if dur < min_dur or dur > max_dur:
            skipped["duration"] += 1
            continue

        if total_sec + dur > max_seconds and rows:
            break

        clip_id = f"hesham_{len(rows):05d}"
        out = RAW_DIR / f"{clip_id}.wav"
        sf.write(out, wav, target_sr)

        rows.append({
            "id": clip_id,
            "wav_path": str(out.relative_to(ROOT)),
            "text": text,
            "duration_sec": round(dur, 3),
            "source_dataset": DATASET_ID,
        })
        total_sec += dur
        if total_sec >= max_seconds:
            break

    pbar.close()

    if not rows:
        raise RuntimeError(f"No clips passed filters. Skipped: {skipped}")

    meta = pd.DataFrame(rows)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    meta.to_csv(output_csv, index=False)

    summary = {
        "clips": len(meta),
        "hours": round(meta["duration_sec"].sum() / 3600, 3),
        "avg_dur": round(meta["duration_sec"].mean(), 2),
        "skipped": skipped,
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f"Saved -> {output_csv}")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--output", default=str(ROOT / "data" / "metadata.csv"))
    p.add_argument("--max-hours", type=float, default=1.0)
    p.add_argument("--min-dur", type=float, default=2.0)
    p.add_argument("--max-dur", type=float, default=12.0)
    a = p.parse_args()
    download_filtered(
        Path(a.output), max_hours=a.max_hours,
        min_dur=a.min_dur, max_dur=a.max_dur,
    )

In [ ]:
%cd /kaggle/working/Sarj

import shutil
from pathlib import Path

# only clean download dirs (keep fixed scripts)
for p in ["data/raw", "data/processed", "data/durations", "data/metadata.csv",
          "data/train.csv", "data/val.csv"]:
    if Path(p).is_dir():
        shutil.rmtree(p, ignore_errors=True)
    elif Path(p).exists():
        Path(p).unlink()

!python data/download_hesham.py --max-hours 1.0 --min-dur 2.0 --max-dur 12.0

In [ ]:
import io, soundfile as sf
from datasets import load_dataset

ds = load_dataset("HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel", split="train", streaming=True)
row = next(iter(ds))
print("audio keys:", row["audio"].keys() if isinstance(row["audio"], dict) else type(row["audio"]))
print("duration:", row.get("audio_duration_s"))
print("text:", row["text"][:60])

if row["audio"].get("bytes"):
    wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]))
    print("decoded OK:", len(wav), "samples @", sr, "Hz")

In [ ]:
# Cell 8 — preprocess
import yaml
from pathlib import Path
cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["audio"]["max_duration"] = 15.0
cfg["audio"]["min_duration"] = 1.5
cfg["train"]["batch_size"] = 8
cfg["train"]["num_workers"] = 0
Path("configs/hesham.yaml").write_text(yaml.dump(cfg, sort_keys=False))
!python data/preprocess.py --config configs/hesham.yaml

In [ ]:
# Cell 10 — train
import yaml, shutil
from pathlib import Path
cfg = yaml.safe_load(open("configs/hesham.yaml"))
cfg["train"]["max_steps"] = 19000
cfg["train"]["lr"] = 0.0003
cfg["train"]["val_every"] = 2000
cfg["train"]["save_every"] = 2000
cfg["train"]["log_every"] = 200
Path("configs/train_hesham.yaml").write_text(yaml.dump(cfg, sort_keys=False))
shutil.rmtree("checkpoints", ignore_errors=True)
Path("checkpoints").mkdir(exist_ok=True)
!python train.py --config configs/train_hesham.yaml --device cuda

In [ ]:
import subprocess
from IPython.display import Audio, display

phrases = [
    "السَّلَامُ عَلَيْكُمْ",
    "كَيْفَ حَالُكَ الْيَوْمَ",
    "أُحِبُّ تَعَلُّمَ اللُّغَةِ الْعَرَبِيَّةِ",
    "شُكْرًا جَزِيلًا لَكُمْ",
]

for i, text in enumerate(phrases):
    out = f"samples/hesham_{i}.wav"
    subprocess.run([
        "python", "synthesize.py",
        "--text", text, "--out", out,
        "--checkpoint", "checkpoints/best.pt",
        "--config", "configs/train_hesham.yaml",
        "--device", "cuda",
    ], check=True)
    print(f"\n=== {text} ===")
    display(Audio(out))

In [ ]:
import subprocess
from IPython.display import Audio, display

phrases = [
    "أَنَا بِخَيْرٍ شُكْرًا",        # I'm fine, thank you
    "صَبَاحُ الخَيْرِ يَا صَدِيقِي",  # Good morning, my friend
    "هَذَا يَوْمٌ جَمِيلٌ",          # This is a beautiful day
    "أُحِبُّ تَعَلُّمَ اللُّغَةِ",    # I love learning the language
    "وَاحِدٌ اثْنَانِ ثَلَاثَةٌ",     # One two three
]
for i, text in enumerate(phrases):
    out = f"samples/hesham_{i}.wav"
    subprocess.run([
        "python", "synthesize.py",
        "--text", text, "--out", out,
        "--checkpoint", "checkpoints/best.pt",
        "--config", "configs/train_hesham.yaml",
        "--device", "cuda",
    ], check=True)
    print(f"\n=== {text} ===")
    display(Audio(out))

In [ ]:
%%writefile scripts/diagnose.py
"""Compare GT mel vs model mel vs original wav."""
from __future__ import annotations
import argparse, sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, soundfile as sf, yaml, torch
from data.audio_utils import mel_to_wav
from data.text_normalize import load_vocab, text_to_ids
from synthesize import load_model

def diagnose(config_path, checkpoint, row_idx=0, out_dir="samples/diag"):
    cfg = yaml.safe_load(open(config_path))
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    sr = cfg["audio"]["sample_rate"]
    n_iters = cfg["synthesis"].get("griffin_lim_iters", 64)

    row = pd.read_csv(ROOT / "data/train.csv").iloc[row_idx]
    text = row["text"]
    gt_mel = np.load(ROOT / row["mel_path"])

    def save_mel(mel, name):
        wav = mel_to_wav(mel, cfg, n_iters)
        peak = np.max(np.abs(wav)) or 1.0
        wav = wav / peak * 0.95
        path = out / name
        sf.write(path, wav, sr)
        return path

    p_a = save_mel(gt_mel, "A_gt_mel.wav")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, _ = load_model(Path(checkpoint), device)
    ids = text_to_ids(text, load_vocab(ROOT / "data/vocab.json"), add_eos=True)
    tokens = torch.tensor([ids], device=device)
    lengths = torch.tensor([len(ids)], device=device)
    with torch.no_grad():
        pred = model.infer(tokens, lengths).cpu().numpy()

    p_b = save_mel(pred, "B_model_same_text.wav")

    orig = ROOT / row["wav_path"]
    p_c = out / "C_original.wav"
    if orig.exists():
        sf.write(p_c, *sf.read(str(orig)))

    t = min(pred.shape[1], gt_mel.shape[1])
    l1 = float(np.mean(np.abs(pred[:, :t] - gt_mel[:, :t])))

    print("Text:", text[:100])
    print(f"Mel L1 (pred vs gt): {l1:.3f}  (lower = better, aim < 0.5)")
    print("A:", p_a, "— vocoder ceiling")
    print("B:", p_b, "— model on same text")
    print("C:", p_c, "— studio target")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--config", default="configs/train_hesham.yaml")
    p.add_argument("--checkpoint", default="checkpoints/best.pt")
    p.add_argument("--row", type=int, default=0)
    args = p.parse_args()
    diagnose(args.config, args.checkpoint, args.row)

In [ ]:
%%writefile synthesize_v2.py
"""Synthesize with mel clamping + stronger Griffin-Lim."""
from __future__ import annotations
import argparse
from pathlib import Path
import numpy as np, soundfile as sf, torch, yaml
from data.audio_utils import mel_to_wav
from data.text_normalize import load_vocab, text_to_ids
from synthesize import load_model

ROOT = Path(__file__).resolve().parent

def clamp_mel(mel: np.ndarray, lo=-5.5, hi=1.5) -> np.ndarray:
    return np.clip(mel, lo, hi)

def synthesize(text, checkpoint, config_path, output, device_name="cuda"):
    cfg = yaml.safe_load(open(config_path))
    device = torch.device(device_name if torch.cuda.is_available() else "cpu")
    model, ckpt_cfg = load_model(checkpoint, device)
    ids = text_to_ids(text, load_vocab(ROOT / "data/vocab.json"), add_eos=True)
    tokens = torch.tensor([ids], dtype=torch.long, device=device)
    lengths = torch.tensor([len(ids)], dtype=torch.long, device=device)

    with torch.no_grad():
        mel = model.infer(tokens, lengths).cpu().numpy()

    mel = clamp_mel(mel)
    n_iters = cfg["synthesis"].get("griffin_lim_iters", 128)
    wav = mel_to_wav(mel, ckpt_cfg, n_iters)
    peak = np.max(np.abs(wav)) or 1.0
    wav = wav / peak * 0.95

    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output, wav, ckpt_cfg["audio"]["sample_rate"])
    print(f"Saved {output} ({len(wav)/ckpt_cfg['audio']['sample_rate']:.2f}s)")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--text", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--checkpoint", default="checkpoints/best.pt")
    p.add_argument("--config", default="configs/train_hesham.yaml")
    p.add_argument("--device", default="cuda")
    a = p.parse_args()
    synthesize(a.text, Path(a.checkpoint), Path(a.config), Path(a.out), a.device)

In [ ]:
%%writefile scripts/generate_samples.py
"""Batch-generate sample wavs."""
import subprocess, sys
from pathlib import Path

PHRASES = [
    ("01_salam", "السَّلَامُ عَلَيْكُمْ"),
    ("02_how", "كَيْفَ حَالُكَ الْيَوْمَ"),
    ("03_arabic", "أُحِبُّ تَعَلُّمَ اللُّغَةِ الْعَرَبِيَّةِ"),
    ("04_thanks", "شُكْرًا جَزِيلًا لَكُمْ"),
    ("05_tech", "تُسَاعِدُ أَدَواتُ تَطْوِيرِ البرمَجَاتِ فِي تَسْهِيلِ العَمَلِ"),
]

def main():
    out_dir = Path("samples/batch")
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt = "checkpoints/best.pt"
    cfg = "configs/train_hesham.yaml"
    for name, text in PHRASES:
        out = out_dir / f"{name}.wav"
        subprocess.run([
            sys.executable, "synthesize_v2.py",
            "--text", text, "--out", str(out),
            "--checkpoint", ckpt, "--config", cfg, "--device", "cuda",
        ], check=True)
        print(f"OK {name}: {text[:40]}")

if __name__ == "__main__":
    main()

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(open("configs/hesham.yaml"))

# DATA / MODEL
cfg["audio"]["max_duration"] = 15.0
cfg["model"]["dropout"] = 0.15          # reduce overfit
cfg["model"]["use_postnet"] = True

# TRAINING — resume-friendly
cfg["train"]["batch_size"] = 4          # better generalization than 8
cfg["train"]["lr"] = 0.0001             # fine-tune (was 0.0003)
cfg["train"]["weight_decay"] = 0.0001
cfg["train"]["max_steps"] = 40000       # continue from ~19k
cfg["train"]["val_every"] = 2000
cfg["train"]["save_every"] = 2000
cfg["train"]["log_every"] = 200
cfg["train"]["num_workers"] = 0

# SYNTHESIS
cfg["synthesis"]["griffin_lim_iters"] = 128

Path("configs/optimized.yaml").write_text(yaml.dump(cfg, sort_keys=False))
print("Saved configs/optimized.yaml")

In [ ]:
%cd /kaggle/working/Sarj
!python train.py --config configs/optimized.yaml --device cuda --resume checkpoints/latest.pt

In [ ]:
# Diagnose
!python scripts/diagnose.py --config configs/optimized.yaml --checkpoint checkpoints/best.pt

# Batch samples (uses synthesize_v2)
!python scripts/generate_samples.py

from IPython.display import Audio, display
import os
for f in sorted(os.listdir("samples/batch")):
    if f.endswith(".wav"):
        print(f)
        display(Audio(f"samples/batch/{f}"))

In [ ]:
%cd /kaggle/working/Sarj

import numpy as np
import pandas as pd
import soundfile as sf
import yaml
import torch
from pathlib import Path
from data.audio_utils import mel_to_wav
from synthesize import load_model
from data.text_normalize import load_vocab, text_to_ids

cfg = yaml.safe_load(open("configs/optimized.yaml"))
row = pd.read_csv("data/train.csv").iloc[0]
text = row["text"]
gt_mel = np.load(row["mel_path"])

def save_mel(mel, path, n_iters=128):
    wav = mel_to_wav(mel, cfg, n_iters)
    wav = wav / (np.max(np.abs(wav)) + 1e-8) * 0.95
    sf.write(path, wav, cfg["audio"]["sample_rate"])
    return path

Path("samples/diag").mkdir(parents=True, exist_ok=True)

# A — GT mel
p_a = save_mel(gt_mel, "samples/diag/A_gt_mel.wav")

# B — model mel (use latest.pt!)
device = torch.device("cuda")
ckpt_path = Path("checkpoints/latest.pt")
if not ckpt_path.exists():
    ckpt_path = Path("checkpoints/best.pt")

model, _ = load_model(ckpt_path, device)
ids = text_to_ids(text, load_vocab(Path("data/vocab.json")), add_eos=True)
tokens = torch.tensor([ids], device=device)
lengths = torch.tensor([len(ids)], device=device)

with torch.no_grad():
    pred_mel = model.infer(tokens, lengths).cpu().numpy()

p_b = save_mel(pred_mel, "samples/diag/B_model_same_text.wav")

# C — original
orig = Path(row["wav_path"])
if orig.exists():
    sf.write("samples/diag/C_original.wav", *sf.read(str(orig)))

# Safe mel comparison — align to shorter length
t = min(pred_mel.shape[1], gt_mel.shape[1])
mel_l1 = float(np.mean(np.abs(pred_mel[:, :t] - gt_mel[:, :t])))

print("Checkpoint:", ckpt_path)
print("GT frames:", gt_mel.shape[1], "| Pred frames:", pred_mel.shape[1], "| Compared:", t)
print("Mel L1 diff:", round(mel_l1, 3), "  (aim < 0.5)")
print("A = GT mel (vocoder limit):", p_a)
print("B = model same text:", p_b)
print("Text:", text[:80])

In [ ]:
from IPython.display import Audio, display

print("A = GT mel (vocoder ceiling)")
display(Audio("samples/diag/A_gt_mel.wav"))

print("B = model on SAME text (should be close to A now)")
display(Audio("samples/diag/B_model_same_text.wav"))

print("C = original studio recording")
display(Audio("samples/diag/C_original.wav"))

In [ ]:
import shutil
shutil.copy("checkpoints/latest.pt", "checkpoints/use_this.pt")
print("Always use checkpoints/latest.pt (step 40000)")

In [ ]:
import subprocess
from IPython.display import Audio, display

phrases = [
    "السَّلَامُ عَلَيْكُمْ",
    "كَيْفَ حَالُكَ الْيَوْمَ",
    "أُحِبُّ تَعَلُّمَ اللُّغَةِ الْعَرَبِيَّةِ",
]

for i, text in enumerate(phrases):
    out = f"samples/final_{i}.wav"
    subprocess.run([
        "python", "synthesize_v2.py",
        "--text", text, "--out", out,
        "--checkpoint", "checkpoints/latest.pt",
        "--config", "configs/optimized.yaml",
        "--device", "cuda",
    ], check=True)
    print(text)
    display(Audio(out))

In [ ]:
%cd /kaggle/working/Sarj

import subprocess
from pathlib import Path
from IPython.display import Audio, display

PHRASES = [
    ("01_salam", "السَّلَامُ عَلَيْكُمْ"),
    ("02_how", "كَيْفَ حَالُكَ الْيَوْمَ"),
    ("03_arabic", "أُحِبُّ تَعَلُّمَ اللُّغَةِ الْعَرَبِيَّةِ"),
    ("04_thanks", "شُكْرًا جَزِيلًا لَكُمْ"),
    ("05_tech", "تُسَاعِدُ أَدَواتُ تَطْوِيرِ البرمَجَاتِ فِي تَسْهِيلِ العَمَلِ"),
]

Path("samples/batch").mkdir(parents=True, exist_ok=True)

# use latest.pt (step 40000) — NOT old best.pt
checkpoint = "checkpoints/latest.pt"
if not Path(checkpoint).exists():
    checkpoint = "checkpoints/best.pt"

config = "configs/optimized.yaml"
if not Path(config).exists():
    config = "configs/train_hesham.yaml"

print("Checkpoint:", checkpoint)
print("Config:", config)
print("-" * 50)

for name, text in PHRASES:
    out = f"samples/batch/{name}.wav"

    # try synthesize_v2 first, fallback to synthesize.py
    script = "synthesize_v2.py" if Path("synthesize_v2.py").exists() else "synthesize.py"
    subprocess.run([
        "python", script,
        "--text", text,
        "--out", out,
        "--checkpoint", checkpoint,
        "--config", config,
        "--device", "cuda",
    ], check=True)

    print(f"\n=== {name} ===")
    print(text)
    display(Audio(out))